In [8]:
import os
import json
import base64
import numpy as np
import pandas as pd
import geopandas as gpd
from typing import Optional
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThanOrEqual
from pyiceberg.catalog import load_catalog

In [9]:
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree

def search_spatial_candidates(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    k: int = 100,
    max_dist: float = 1000, 
    id_col: str = "id",
    crs_for_distance: int = 3857,
):
    """
    Attach k nearest compared POI ids & distances to reference_gdf.

    Returns
    -------
    GeoDataFrame with two new columns:
    - cand_ids   : list of compared ids
    - cand_dist_m: list of distances (meters)
    """

    ref_proj = reference_gdf.to_crs(crs_for_distance)
    cmp_proj = compared_gdf.to_crs(crs_for_distance)

    ref_xy = np.column_stack([ref_proj.geometry.x, ref_proj.geometry.y])
    cmp_xy = np.column_stack([cmp_proj.geometry.x, cmp_proj.geometry.y])

    tree = cKDTree(cmp_xy)
    k_eff = min(k, len(compared_gdf))

    dist, idx = tree.query(ref_xy, k=k_eff)

    if k_eff == 1:
        dist = dist.reshape(-1, 1)
        idx = idx.reshape(-1, 1)

    cmp_ids = compared_gdf[id_col].to_numpy()

    cand_ids = []
    cand_dists = []

    for row_idx, row_dist in zip(idx, dist):
        ids = []
        dists = []

        for j, d in zip(row_idx, row_dist):
            if d <= max_dist:
                ids.append(cmp_ids[j])
                dists.append(d)

        cand_ids.append(ids)
        cand_dists.append(dists)

    result = reference_gdf.copy()
    result["cand_ids"] = cand_ids
    result["cand_dist_m"] = cand_dists

    return result

In [10]:
FOOD_WORDS = {
    "RESTAURANT","RESTURANT","RESTARAUNT",
    "CAFE","CAFÉ","COFFEE","BAR","BISTRO","GRILL",
    "KITCHEN","DINER","EATERY","STEAKHOUSE",
    "SEAFOOD","BUFFET","BBQ","PIZZA","SUSHI","RAMEN",
    "NOODLE","NOODLES","BURGER","BURGERS","TACO","TACOS",
    "CHICKEN","WINGS","BAKERY","DELI","DELICATESSEN",
    "COURT","FOOD","EXPRESS","HOUSE","SHOP"
}

PLACE_WORDS = {
    "HALL","CENTER","CENTRE","PLAZA","MARKET","MALL",
    "GARDEN","GARDENS","PARK","SQUARE","TOWER","TOWERS",
    "STATION","TERMINAL","BUILDING","GALLERY","THEATER","SCHOOL","COURT","INN",
    "HOTEL","MOTEL","INN","SUITES","SUITE",
    "SPA","SALON","STUDIO","CENTER","CENTRE",
    "CLUB","LOUNGE","STATION","STOP"
}

LEGAL_WORDS = {
    "LLC","INC","CORP","CORPORATION","CO","COMPANY",
    "LTD","LIMITED","GROUP","HOLDINGS","OFFICE"
}

GRAMMAR = {
    "THE","OF","AT","AND","FOR","IN","ON","BY","WITH","TO","FROM"
}

NON_PRIMARY_TOKENS = FOOD_WORDS | PLACE_WORDS | LEGAL_WORDS | GRAMMAR

In [11]:
from rapidfuzz import process, fuzz
import pandas as pd
import re
import unicodedata


def clean_name(s):
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)) # 1. unicode normalize (remove accents)
    s = s.upper() # 2. uppercase
    s = re.sub(r"\([^)]*\)", "", s) 
    s = re.sub(r"\b'S\b", "", s) # new change
    s = re.sub(r"\bS\b", "", s) # new change
    s = s.encode("ascii", errors="ignore").decode() # 4. remove emoji / non ascii
    s = re.sub(r"[^\w\s]", " ", s) # 5. replace special chars with space
    s = re.sub(r"\s+", " ", s) # 6. collapse spaces

    return s.strip()

def extract_prinmary_str(name):

    tokens = name.split()
    core = [t for t in tokens if t not in NON_PRIMARY_TOKENS]
    if len(core) == 1 and len(core[0]) < 3:
        return name
    if core:
        return " ".join(core)
    return name

def match_by_name(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    re_name_col: str = "name",
    comp_name_col: str = "name",
    comp_id: str = "id",
    comp_id_col: str="cat_main",
    threshold: int = 80,
):
    """
    Perform WRatio name matching within spatial candidates.

    Returns
    -------
    GeoDataFrame with:
    - matched_id_name
    - name_score
    """

    # clean names for matching
    id_to_name_clean = compared_gdf.set_index(comp_id)[comp_name_col].apply(clean_name).apply(extract_prinmary_str).to_dict()
    # raw names for storage
    id_to_name_raw = compared_gdf.set_index(comp_id)[comp_name_col].to_dict()
    # raw compared df category
    id_to_cat = compared_gdf.set_index(comp_id)[comp_id_col].to_dict()

    matched_ids = []
    scores = []
    loc_dists = []
    matched_names = []
    matched_cats = []

    for _, row in reference_gdf.iterrows():
        query = extract_prinmary_str(clean_name(row.get(re_name_col)))

        if not isinstance(query, str) or not row["cand_ids"]:
            matched_ids.append(pd.NA)
            scores.append(pd.NA)
            loc_dists.append(pd.NA)
            matched_names.append(pd.NA)
            matched_cats.append(pd.NA)
            continue

        cand_names = [id_to_name_clean.get(cid, "") for cid in row["cand_ids"]]

        top_matches = process.extract(
            query,
            cand_names,
            scorer=fuzz.WRatio,
            limit=5
        )

        best_score = -1
        best_pos = None

        for name, wr, pos in top_matches:

            pr = fuzz.partial_ratio(query, name)
            ts = fuzz.token_sort_ratio(query, name)
            tset = fuzz.token_set_ratio(query, name)

            combined = max(wr, pr, ts, tset)

            if combined > best_score:
                best_score = combined
                best_pos = pos

        score = best_score

        if score >= threshold:
            matched_ids.append(row["cand_ids"][best_pos])
            scores.append(score)
            loc_dists.append(row["cand_dist_m"][best_pos])
            matched_names.append(id_to_name_raw.get(row["cand_ids"][best_pos]))
            matched_cats.append(id_to_cat.get(row["cand_ids"][best_pos]))
        else:
            matched_ids.append(pd.NA)
            scores.append(score)
            loc_dists.append(pd.NA)
            matched_names.append(pd.NA)
            matched_cats.append(pd.NA)


    result = reference_gdf.copy()
    result["matched_id"] = matched_ids
    result["name_score"] = scores
    result["location_distance"] = loc_dists
    result["matched_name"] = matched_names
    result["matched_cat_main"] = matched_cats

    return result

In [12]:
import pandas as pd
import geopandas as gpd
from rapidfuzz import fuzz
import re
import unicodedata

def clean_name(s):
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)) # 1. unicode normalize (remove accents)
    s = s.upper() # 2. uppercase
    s = re.sub(r"\([^)]*\)", "", s) 
    s = s.encode("ascii", errors="ignore").decode() # 4. remove emoji / non ascii
    s = re.sub(r"[^\w\s]", " ", s) # 5. replace special chars with space
    s = re.sub(r"\s+", " ", s) # 6. collapse spaces

    return s.strip()

def address_score_check(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    addr_col_ref: str = "addr_simple",
    addr_col_cmp: str = "address",
    matched_id_col: str = "matched_id",
    id_col: str = "id",
    out_col: str = "address_score",
    out_addr_col: str = "matched_address", 
):
    """
    Compute address similarity score (0-100) for already-matched rows.

    Only rows with non-null `matched_id_col` will be scored.
    Others will have NaN.

    Returns
    -------
    GeoDataFrame with a new column `out_col`.
    """

    # map: compared id -> compared address
    id_to_addr_clean = compared_gdf.set_index(id_col)[addr_col_cmp].apply(clean_name).to_dict()
    id_to_addr_raw = compared_gdf.set_index(id_col)[addr_col_cmp].to_dict()

    scores = []
    matched_addrs = []

    for _, row in reference_gdf.iterrows():
        matched_id = row.get(matched_id_col)

        # no matched id -> no score
        if pd.isna(matched_id):
            scores.append(pd.NA)
            matched_addrs.append(pd.NA)
            continue

        addr_ref = clean_name(row.get(addr_col_ref))
        addr_cmp = id_to_addr_clean.get(matched_id)

        if isinstance(addr_cmp, str) and addr_cmp.strip():
            matched_addrs.append(id_to_addr_raw.get(matched_id))
        else:
            matched_addrs.append(pd.NA)

        # missing address on either side -> no score
        if not isinstance(addr_ref, str) or not addr_ref.strip():
            scores.append(pd.NA)
            continue
        if not isinstance(addr_cmp, str) or not addr_cmp.strip():
            scores.append(pd.NA)
            continue

        wr = fuzz.WRatio(addr_ref, addr_cmp)
        pr = fuzz.partial_ratio(addr_ref, addr_cmp)
        ts = fuzz.token_sort_ratio(addr_ref, addr_cmp)
        tset = fuzz.token_set_ratio(addr_ref, addr_cmp)

        scores.append(int(max(wr, pr, ts, tset)))

    result = reference_gdf.copy()
    result[out_col] = scores
    result[out_addr_col] = matched_addrs
    return result

In [13]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

def calculate_similarity_check(
    df, 
    cat_col_ref: str = "primary_type", 
    cat_col_cmp: str = "matched_cat_main", 
    id_col: str = "matched_id", 
    result_col: str =  "category_sim",
):

    # 1. Setup Device
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    
    # 2. Primary Gatekeeper: matched_id must be present
    mask = df[id_col].notna() & (df[id_col].astype(str).str.strip() != "")
    
    # 3. Create a helper to handle potential Nulls in text columns
    temp_df = df[mask].copy()
    
    # Identify where text is actually missing within the masked rows
    text_missing_mask = temp_df[cat_col_ref].isna() | temp_df[cat_col_cmp].isna()
    
    # Fill NaNs with empty strings just for the encoding step
    texts_1 = temp_df[cat_col_ref].fillna("").astype(str).tolist()
    texts_2 = temp_df[cat_col_cmp].fillna("").astype(str).tolist()

    print(f"Encoding {len(temp_df)} rows...")

    # 4. Batch Encoding
    emb1 = model.encode(texts_1, batch_size=256, show_progress_bar=True, convert_to_tensor=True)
    emb2 = model.encode(texts_2, batch_size=256, show_progress_bar=True, convert_to_tensor=True)

    # 5. Calculate Similarity
    scores = torch.nn.functional.cosine_similarity(emb1, emb2, dim=1).cpu().numpy()
    
    # 6. Apply NaN to rows where text was missing
    # Even if we encoded an empty string, the result isn't "real" if data was missing
    scores[text_missing_mask.values] = np.nan

    # 7. Map back to original dataframe
    df[result_col] = np.nan
    df.loc[mask, result_col] = scores
    
    return df

In [14]:
chicago_gplc = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_msa_poi/chicago_gplc.geojson')
chicago_gplc

,circle_id,id,name,address,primary_type,lat,lon,business_status,primary_cat,addr_simple,naics_code,naics_definition,geometry
0,0,ChIJd1-a836IDYgR6PolD_KgODE,Trudeau's Body Shop,"515 N Clark St, Sheldon, IL 60966, USA",car_repair,40.775277,-87.575841,OPERATIONAL,automotive,515 N Clark St,8111.0,Automotive Repair and Maintenance,POINT (-87.57584 40.77528)
1,0,ChIJRc-9WACJDYgRVZb9tKwxdyc,Village of Sheldon Administration Building Par...,"Sheldon, IL 60966, USA",parking_lot,40.773454,-87.563814,OPERATIONAL,automotive,Sheldon,8123.0,Parking Lots and Garages,POINT (-87.56381 40.77345)
2,1,ChIJh1H7CHN6EogRq3yq0ddXj9k,BP,"409 S 7th St, Kentland, IN 47951, USA",gas_station,40.761758,-87.438293,OPERATIONAL,automotive,409 S 7th St,447.0,Gasoline Stations,POINT (-87.43829 40.76176)
3,1,ChIJ8_MdS55wEogRyr7tM4FNZyA,Marathon Gas,"102 N 7th St, Kentland, IN 47951, USA",gas_station,40.768374,-87.438839,OPERATIONAL,automotive,102 N 7th St,447.0,Gasoline Stations,POINT (-87.43884 40.76837)
4,1,ChIJzaSgXgd6EogRNZrXaY3d98A,Model 1 Chevrolet of Kentland,"308 W Seymour St, Kentland, IN 47951, USA",car_dealer,40.768603,-87.454434,OPERATIONAL,automotive,308 W Seymour St,441.0,Motor Vehicle and Parts Dealers,POINT (-87.45443 40.76860)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
56956,428,ChIJJ32HgQ3zD4gRo0W2SYFi0ro,Francis Energy Charging Station,"39131 N Sheridan Rd, Beach Park, IL 60087, USA",electric_vehicle_charging_station,42.429671,-87.825194,OPERATIONAL,transportation,39131 N Sheridan Rd,8111.0,Automotive Repair and Maintenance,POINT (-87.82519 42.42967)
56957,428,ChIJcfhLpBT1D4gRyyucDs7YN8Y,Beach Ridge Day Use Area Parking Lot,"Zion, IL 60099, USA",parking_lot,42.468193,-87.799932,OPERATIONAL,transportation,Zion,8123.0,Parking Lots and Garages,POINT (-87.79993 42.46819)
56958,428,ChIJAbu4aX7zD4gRNF4MA6GYmmw,"Zion,IL Metra Station West Parking Lot","Zion, IL 60099, USA",parking_lot,42.449095,-87.818332,OPERATIONAL,transportation,Zion,8123.0,Parking Lots and Garages,POINT (-87.81833 42.44910)
56959,428,ChIJJ7TMedP1D4gRiLMElMLB6Fg,Bison Pallets,"2415 23rd St, Zion, IL 60099, USA",transportation_service,42.453897,-87.845944,OPERATIONAL,transportation,2415 23rd St,4859.0,Other Transit and Ground Passenger Transportation,POINT (-87.84594 42.45390)


In [8]:
import json
import pandas as pd

def safe_json_load(x):
    try:
        return json.loads(x) if isinstance(x, str) else x
    except:
        return None

def extract_categories(x):
    data = safe_json_load(x)
    if isinstance(data, dict):
        primary = data.get('primary')
        alternate_list = data.get('alternate', [])
        alternate_str = ", ".join(alternate_list) if alternate_list else None
        return primary, alternate_str
    return None, None

def extract_name(x):
    data = safe_json_load(x)
    if isinstance(data, dict):
        return data.get('primary')
    return None

def extract_address(x):
    data = safe_json_load(x)
    if isinstance(data, list) and len(data) > 0:
        return data[0].get('freeform')
    return None

def extract_sources(x):
    data = safe_json_load(x)
    if isinstance(data, list) and len(data) > 0:
        first = data[0] 
        return first.get('dataset'), first.get('record_id'), first.get('update_time')
    return None, None, None


bspo_ove = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo msa/bspo_ove.csv')

bspo_ove[['cat_main', 'cat_alt']] = bspo_ove['categories'].apply(
    lambda x: pd.Series(extract_categories(x))
)

bspo_ove['name'] = bspo_ove['names'].apply(extract_name)
bspo_ove['address'] = bspo_ove['addresses'].apply(extract_address)
bspo_ove[['src_dataset', 'src_record_id', 'src_update_time']] = bspo_ove['sources'].apply(lambda x: pd.Series(extract_sources(x)))


bspo_ove['WKB'] = bspo_ove['geometry'].apply(lambda x: x[2:])
geometry = gpd.GeoSeries.from_wkb(bspo_ove['WKB'], crs=4326)

bspo_ove = gpd.GeoDataFrame(bspo_ove, geometry=geometry)
bspo_ove = bspo_ove[['id','name','address','cat_main','cat_alt','confidence','operating_status','version','src_dataset','src_update_time','geometry']]
bspo_ove

,id,name,address,cat_main,cat_alt,confidence,operating_status,version,src_dataset,src_update_time,geometry
0,d44807ab-9820-427c-84b1-eac2e1c9cb1b,"Lake Leland Quilcene, Wa",None,lake,"active_life, sports_club_and_league",0.323479,open,2,meta,2025-12-01T08:00:00.000Z,POINT (-122.73965 47.96577)
1,77c81091-7d93-476f-9298-9f68fde97b0f,Kodama Farm and Food Forest,42 Tall Tree Ln,farm,"livestock_breeder, urban_farm",0.950063,open,7,meta,2025-12-01T08:00:00.000Z,POINT (-122.74099 47.96374)
2,3aeb2bf6-01e3-4263-a66b-a4ca82fef297,Egg and I Fuchsias,2027 Egg and I Rd,farm,"breakfast_and_brunch_restaurant, cafe",0.917024,open,5,meta,2025-12-01T08:00:00.000Z,POINT (-122.75817 47.95798)
3,3da06df0-7ef2-455f-9603-e58400207494,Phillips Painting,1691 Egg And I Rd,painting,None,0.947100,open,4,Microsoft,2025-05-29T06:47:15.977Z,POINT (-122.76448 47.95435)
4,bb303c91-e532-4f97-ba5b-406372fdee4d,Harmonys Way Farm,1460 Egg and I Rd,farm,"vitamins_and_supplements, insurance_agency",0.639236,open,4,meta,2025-12-01T08:00:00.000Z,POINT (-122.77004 47.95224)
...,...,...,...,...,...,...,...,...,...,...,...
12439,fc313c3b-949c-45f3-988b-59a40a2c236a,Mount Walker,Mount Walker,landmark_and_historical_building,"park, hiking_trail",0.639236,open,1,meta,2025-12-01T08:00:00.000Z,POINT (-122.90684 47.78887)
12440,060bb2a5-2553-4745-913c-09435773e188,Devils Lake,None,lake,"attractions_and_activities, landmark_and_histo...",0.950063,open,5,meta,2025-12-01T08:00:00.000Z,POINT (-122.88288 47.79234)
12441,9daec706-a795-443e-ad44-d6cca11badbf,Marmot Pass,None,hiking_trail,sporting_goods,0.639236,open,5,meta,2025-12-01T08:00:00.000Z,POINT (-122.91249 47.81141)
12442,5b80ea90-d852-4a48-b80b-09e0142ed578,Quilcene National Fish Hatchery,281 Fish Hatchery Rd,environmental_conservation_organization,"public_and_government_association, public_serv...",0.882999,open,5,meta,2025-12-01T08:00:00.000Z,POINT (-122.91698 47.80821)


In [15]:
chicago_ove = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_msa_poi/chicago_ove.geojson', driver='GeoJSON')
chicago_ove

,id,name,address,cat_main,cat_alt,confidence,operating_status,version,src_dataset,src_update_time,geometry
0,8b891dae-b4b8-41c0-ba71-f28d87d9ff05,"Alisha Jackson, APRN",1614 E Norris Dr,health_consultant,"nurse_practitioner, active_life, health_and_me...",0.770000,open,5,Microsoft,2024-05-04T11:17:09.623000+00:00,POINT (-88.81476 41.35163)
1,999bc680-4df9-4e0d-9d1c-62a3a022564d,River Heights,NaN,river,NaN,0.965772,open,4,meta,2026-03-09T00:00:00+00:00,POINT (-88.77150 41.91814)
2,c4d3f1fe-c5e9-41cc-b9c7-87c04aeb7652,Mobil,930 Annie Glidden Road,gas_station,automotive,0.947100,open,5,Microsoft,2025-10-01T02:04:51.882999+00:00,POINT (-88.77285 41.94492)
3,01b00cdc-2e70-498b-b1bc-deb1f0bf026f,"Cub Scout Pack 239 - Geneva, IL",0S350 Grengs Ln,printing_services,youth_organizations,0.936728,open,7,meta,2026-03-09T00:00:00+00:00,POINT (-88.37682 41.86329)
4,48c456b7-f1d1-42a4-925c-5d483a05bd26,Fox River Park,817 St George St,park,"arts_and_entertainment, attractions_and_activi...",0.965772,open,1,meta,2026-03-09T00:00:00+00:00,POINT (-88.83060 41.35204)
...,...,...,...,...,...,...,...,...,...,...,...
433576,617f7f7c-d6e3-4eec-aa4c-f253edc4818b,Picnic Wine & Provisions,7301 N Sheridan Rd,grocery_store,"liquor_store, supermarket",0.974047,open,7,meta,2026-03-09T00:00:00+00:00,POINT (-87.66401 42.01428)
433577,1b7912b0-c8ed-4d99-934b-0cc8f161bd64,W Touhy Ave,4019 W TOUHY AVE,automotive_repair,"car_inspection, roadside_assistance, engine_re...",0.400000,open,3,AllThePlaces,2026-02-02T23:02:23+00:00,POINT (-87.72933 42.01154)
433578,3509e99a-0728-472c-957b-e45526f7214f,Walter Athletics Center,2255 N Campus Dr,sports_and_recreation_venue,"gym, campus_building",0.938874,open,7,meta,2026-03-09T00:00:00+00:00,POINT (-87.67170 42.05918)
433579,06a0a198-4edf-442a-8e3f-7ec0938a49f2,Skokie Banquet and Conference Center,5300 Touhy Ave,event_planning,"topic_concert_venue, caterer",0.972216,open,7,meta,2026-03-09T00:00:00+00:00,POINT (-87.75997 42.01244)


In [16]:
chicago_gplc_ove = search_spatial_candidates(reference_gdf=chicago_gplc, compared_gdf=chicago_ove, k=100)

In [17]:
chicago_gplc_ove = match_by_name(reference_gdf=chicago_gplc_ove, compared_gdf=chicago_ove, re_name_col = "name", comp_name_col = "name", threshold=80)

In [18]:
chicago_gplc_ove = address_score_check(reference_gdf=chicago_gplc_ove, compared_gdf=chicago_ove)

In [19]:
chicago_gplc_ove = calculate_similarity_check(chicago_gplc_ove, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 45550 rows...


Batches:   0%|          | 0/178 [00:00<?, ?it/s]

Batches:   0%|          | 0/178 [00:00<?, ?it/s]

In [20]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score', 'category_sim']
for col in cols_to_fix:
    chicago_gplc_ove[col] = pd.to_numeric(chicago_gplc_ove[col], errors='coerce')

In [ ]:
df = chicago_gplc_ove[chicago_gplc_ove["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','naics_definition','matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo_google_comparison/bsop_gplc_ove_2000_sample.csv', index=False)

In [22]:
chicago_gplc_ove['matched_id'].notna().sum() / len(chicago_gplc_ove)

0.7996699496146487

In [23]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_google_comparison/chicago_gplc_ove_1000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check = df_sample_check.merge(chicago_gplc_ove[['id','name_score','location_distance','address_score','category_sim']], left_on="id", right_on="id", how="left")

In [24]:
df_sample_check['is_true'].value_counts()

is_true
1    931
0     69
Name: count, dtype: int64

In [25]:
df = df_sample_check.copy()

In [26]:
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance', 'address_score', 'category_sim']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [28]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    enable_categorical=False,
    eval_metric='auc',
    random_state=42
)

xgb_clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False  
)

xgb_y_pred = xgb_clf.predict(X_test)
xgb_y_prob = xgb_clf.predict_proba(X_test)[:, 1]

xgb_cr = classification_report(y_test, xgb_y_pred)
xgb_auc = roc_auc_score(y_test, xgb_y_prob)

print(xgb_cr)
print("XGBoost AUC:", xgb_auc)

              precision    recall  f1-score   support

           0       0.93      0.76      0.84        17
           1       0.98      1.00      0.99       233

    accuracy                           0.98       250
   macro avg       0.96      0.88      0.91       250
weighted avg       0.98      0.98      0.98       250

XGBoost AUC: 0.9926786165109821


In [29]:
mask = chicago_gplc_ove['matched_id'].notnull()
df_pred = chicago_gplc_ove.loc[mask].copy()

feature_cols = ['name_score', 'location_distance', 'address_score', 'category_sim']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = xgb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

chicago_gplc_ove.loc[mask, 'is_true_match'] = df_pred['is_true_match']
chicago_gplc_ove.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [30]:
chicago_gplc_ove['is_true_match'].value_counts()

is_true_match
True     40618
False     4932
Name: count, dtype: int64

In [31]:
chicago_gplc_ove.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 56961 entries, 0 to 56960
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   circle_id          56961 non-null  int64   
 1   id                 56961 non-null  object  
 2   name               56961 non-null  object  
 3   address            56961 non-null  object  
 4   primary_type       56961 non-null  object  
 5   lat                56961 non-null  float64 
 6   lon                56961 non-null  float64 
 7   business_status    54054 non-null  object  
 8   primary_cat        56961 non-null  object  
 9   addr_simple        56961 non-null  object  
 10  naics_code         56959 non-null  float64 
 11  naics_definition   56959 non-null  object  
 12  geometry           56961 non-null  geometry
 13  cand_ids           56961 non-null  object  
 14  cand_dist_m        56961 non-null  object  
 15  matched_id         45550 non-null  object  
 

In [32]:
chicago_gplc_ove.drop(columns=['cand_ids','cand_dist_m']).to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_google_comparison/chicago_gplc_ove.geojson', driver='GeoJSON')